# Document Loading & Text Extraction

# 01. Document Loading and Text Extraction

## Objective

Understand how different document formats expose their textual content and investigate the extraction methods required to process them.

## Document Formats

Three document formats are investigated:

1. Plain Text (TXT)
2. Markdown (MD)
3. Portable Document Format (PDF)

## Extraction Methods

| Format | Extraction Method |
|---|---|
| TXT | Python file reading |
| Markdown | Python file reading |
| PDF | PyMuPDF page-level extraction |

## Experimental Procedure

1. Define a common source text.
2. Generate controlled document fixtures.
3. Extract text from each document.
4. Preserve individual PDF page boundaries.
5. Calculate character, word, and line counts.
6. Compare the extracted content with the original source.
7. Inspect differences introduced by document formatting.

## Evaluation Metrics

- Character count
- Word count
- Line count
- Extraction success
- Exact text equality
- Normalized text equality
- PDF page count

## Limitations

The generated documents are controlled experimental fixtures.

The results demonstrate basic document extraction behavior and do not represent extraction quality across arbitrary PDFs, scanned documents, complex tables, or multi-column layouts.

In [2]:
from pathlib import Path
import json
import re

import pymupdf


# ============================================================
# 1. Project Paths
# ============================================================

PROJECT_ROOT = Path.cwd()

EXPERIMENT_DIR = PROJECT_ROOT / "data" / "experiments" / "day1_fixtures"

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. Define Controlled Source Content
# ============================================================

SOURCE_PARAGRAPHS = [
    "HADOTA Local AI Assistant",
    "HADOTA is a local AI assistant powered by Qwen through Ollama.",
    "The application uses FastAPI as its backend framework.",
    (
        "The system supports conversation management, structured outputs, "
        "and local language model inference."
    ),
    "The project is being extended with Retrieval-Augmented Generation.",
    (
        "RAG enables the assistant to retrieve relevant information "
        "from external knowledge sources before generating an answer."
    ),
]

SOURCE_TEXT = "\n\n".join(SOURCE_PARAGRAPHS)


# ============================================================
# 3. Define Document Paths
# ============================================================

TXT_PATH = EXPERIMENT_DIR / "sample.txt"
MD_PATH = EXPERIMENT_DIR / "sample.md"
PDF_PATH = EXPERIMENT_DIR / "sample.pdf"


# ============================================================
# 4. Generate TXT Document
# ============================================================

TXT_PATH.write_text(SOURCE_TEXT, encoding="utf-8")


# ============================================================
# 5. Generate Markdown Document
# ============================================================

MARKDOWN_CONTENT = "# HADOTA Local AI Assistant\n\n" + SOURCE_TEXT

MD_PATH.write_text(MARKDOWN_CONTENT, encoding="utf-8")


# ============================================================
# 6. Generate a Two-Page PDF
# ============================================================

pdf_document = pymupdf.open()

pdf_pages_content = [
    "\n\n".join(SOURCE_PARAGRAPHS[:3]),
    "\n\n".join(SOURCE_PARAGRAPHS[3:]),
]

for page_content in pdf_pages_content:
    page = pdf_document.new_page()

    remaining_space = page.insert_textbox(
        pymupdf.Rect(50, 50, 545, 780), page_content, fontsize=11
    )

    if remaining_space < 0:
        raise RuntimeError("The PDF text does not fit inside the page.")

pdf_document.save(PDF_PATH)
pdf_document.close()


# ============================================================
# 7. Extract TXT Content
# ============================================================


def extract_txt(file_path: Path) -> str:

    return file_path.read_text(encoding="utf-8")


# ============================================================
# 8. Extract Markdown Content
# ============================================================


def extract_markdown(file_path: Path) -> str:

    return file_path.read_text(encoding="utf-8")


# ============================================================
# 9. Extract PDF Content Page by Page
# ============================================================


def extract_pdf(file_path: Path) -> list[dict]:

    extracted_pages = []

    with pymupdf.open(file_path) as document:
        for page_number, page in enumerate(document.pages(), start=1):
            page_text = page.get_text("text", sort=True)

            extracted_pages.append(
                {
                    "page_number": page_number,
                    "text": page_text,
                    "character_count": len(page_text),
                    "is_empty": not page_text.strip(),
                }
            )

    return extracted_pages


# ============================================================
# 10. Load All Documents
# ============================================================

txt_text = extract_txt(TXT_PATH)

markdown_text = extract_markdown(MD_PATH)

pdf_pages = extract_pdf(PDF_PATH)

pdf_text = "\n\n".join(page["text"] for page in pdf_pages)


# ============================================================
# 11. Calculate Text Statistics
# ============================================================


def calculate_statistics(text: str) -> dict:

    return {
        "characters": len(text),
        "words": len(text.split()),
        "lines": len(text.splitlines()),
    }


results = {
    "TXT": calculate_statistics(txt_text),
    "Markdown": calculate_statistics(markdown_text),
    "PDF": calculate_statistics(pdf_text),
}


# ============================================================
# 12. Normalize Whitespace for Comparison
# ============================================================


def normalize_whitespace(text: str) -> str:

    return re.sub(r"\s+", " ", text).strip()


normalized_source = normalize_whitespace(SOURCE_TEXT)

normalized_txt = normalize_whitespace(txt_text)

normalized_markdown = normalize_whitespace(markdown_text)

normalized_pdf = normalize_whitespace(pdf_text)


# ============================================================
# 13. Compare Extracted Content
# ============================================================

comparison = {
    "TXT_exact_match": (txt_text == SOURCE_TEXT),
    "TXT_normalized_match": (normalized_txt == normalized_source),
    "Markdown_contains_source": (SOURCE_TEXT in markdown_text),
    "Markdown_has_heading": (markdown_text.startswith("# ")),
    "PDF_normalized_match": (normalized_pdf == normalized_source),
    "PDF_page_count": len(pdf_pages),
}


# ============================================================
# 14. Display Results
# ============================================================

print("=" * 65)
print("DOCUMENT LOADING AND TEXT EXTRACTION")
print("=" * 65)


print("\n1. GENERATED DOCUMENTS")

for file_path in [TXT_PATH, MD_PATH, PDF_PATH]:
    print({"name": file_path.name, "size_bytes": file_path.stat().st_size})


print("\n2. EXTRACTION STATISTICS")

print(json.dumps(results, indent=4, ensure_ascii=False))


print("\n3. CONTENT COMPARISON")

print(json.dumps(comparison, indent=4))


print("\n4. PDF PAGE INFORMATION")

print(json.dumps(pdf_pages, indent=4, ensure_ascii=False))


print("\n5. TEXT PREVIEW")

print("TXT:")
print(txt_text[:150])

print("\nMARKDOWN:")
print(markdown_text[:150])

print("\nPDF:")
print(pdf_text[:150])


print("\nEXPERIMENT COMPLETED")

DOCUMENT LOADING AND TEXT EXTRACTION

1. GENERATED DOCUMENTS
{'name': 'sample.txt', 'size_bytes': 446}
{'name': 'sample.md', 'size_bytes': 477}
{'name': 'sample.pdf', 'size_bytes': 1636}

2. EXTRACTION STATISTICS
{
    "TXT": {
        "characters": 436,
        "words": 59,
        "lines": 11
    },
    "Markdown": {
        "characters": 465,
        "words": 64,
        "lines": 13
    },
    "PDF": {
        "characters": 436,
        "words": 59,
        "lines": 13
    }
}

3. CONTENT COMPARISON
{
    "TXT_exact_match": true,
    "TXT_normalized_match": true,
    "Markdown_contains_source": true,
    "Markdown_has_heading": true,
    "PDF_normalized_match": true,
    "PDF_page_count": 2
}

4. PDF PAGE INFORMATION
[
    {
        "page_number": 1,
        "text": "HADOTA Local AI Assistant\n\nHADOTA is a local AI assistant powered by Qwen through Ollama.\n\nThe application uses FastAPI as its backend framework.",
        "character_count": 145,
        "is_empty": false
    },
  

## Result Interpretation

Extraction Statistics

| Format | Characters | Words | Lines |
|---|---:|---:|---:|
| TXT | 436 | 59 | 11 |
| Markdown | 465 | 64 | 13 |
| PDF | 436 | 59 | 13 |

The TXT extractor reproduced the original source text exactly.

The Markdown extractor preserved the original content and its additional heading syntax. The character and word counts increased because the Markdown fixture contained an additional heading.

The PDF extractor successfully recovered the source content after whitespace normalization. However, the extracted line count differed because PDF text layout introduced additional line breaks.

The PDF extraction process also preserved page-level information, producing two separate page records.

### Finding

The experiment demonstrated that TXT, Markdown, and PDF require different extraction strategies.

TXT and Markdown can be loaded through standard Python file operations, while PDF requires a dedicated extraction library such as PyMuPDF.

Page-level extraction is important for preserving document provenance and supporting future source attribution.

Whitespace normalization can help compare extracted content, but it must not be confused with general-purpose text normalization.

### Limitations

The experiment used controlled document fixtures with simple textual content.

It did not evaluate scanned PDFs, OCR, complex tables, multi-column layouts, or extraction quality across heterogeneous real-world documents.

# 02. Document Discovery

## Objective

Understand how document discovery identifies candidate files within a knowledge base.

## Key Concepts

- Directory traversal
- Recursive file discovery
- File extensions
- Supported document formats
- Deterministic ordering
- File size inspection

## Experimental Procedure

1. Inspect the existing knowledge base directory.
2. Discover files recursively.
3. Group files by extension.
4. Identify supported and unsupported formats.
5. Calculate file counts and total sizes.
6. Inspect relative file paths.

## Expected Outcome

A structured inventory of the available documents that can be used as input for the ingestion pipeline.

In [4]:
from pathlib import Path
from collections import Counter, defaultdict
import json

CURRENT_DIR = Path.cwd()

PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

KNOWLEDGE_BASE_DIR = PROJECT_ROOT / "knowledge base" / "raw"

print("Current directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Knowledge base:", KNOWLEDGE_BASE_DIR)
print("Directory exists:", KNOWLEDGE_BASE_DIR.exists())


# Supported document formats
SUPPORTED_EXTENSIONS = {".pdf", ".md", ".txt"}


# Discover all files recursively
discovered_files = sorted(
    path for path in KNOWLEDGE_BASE_DIR.rglob("*") if path.is_file()
)


# Group files by extension
files_by_extension = defaultdict(list)

for file_path in discovered_files:
    extension = file_path.suffix.lower()

    files_by_extension[extension].append(file_path)


# Build discovery statistics
extension_counts = Counter(file_path.suffix.lower() for file_path in discovered_files)


supported_files = [
    file_path
    for file_path in discovered_files
    if file_path.suffix.lower() in SUPPORTED_EXTENSIONS
]


unsupported_files = [
    file_path
    for file_path in discovered_files
    if file_path.suffix.lower() not in SUPPORTED_EXTENSIONS
]


total_size = sum(file_path.stat().st_size for file_path in discovered_files)


# Prepare inventory
inventory = {
    "root_directory": str(KNOWLEDGE_BASE_DIR),
    "total_files": len(discovered_files),
    "supported_files": len(supported_files),
    "unsupported_files": len(unsupported_files),
    "total_size_bytes": total_size,
    "extension_counts": dict(extension_counts),
    "files": [
        {
            "name": file_path.name,
            "relative_path": str(file_path.relative_to(PROJECT_ROOT)),
            "extension": file_path.suffix.lower(),
            "size_bytes": file_path.stat().st_size,
            "supported": (file_path.suffix.lower() in SUPPORTED_EXTENSIONS),
        }
        for file_path in discovered_files
    ],
}


print("=" * 65)
print("DOCUMENT DISCOVERY EXPERIMENT")
print("=" * 65)

print(json.dumps(inventory, indent=4, ensure_ascii=False))

Current directory: d:\Projects\Local Mini RAG Document QA System\notebooks
Project root: d:\Projects\Local Mini RAG Document QA System
Knowledge base: d:\Projects\Local Mini RAG Document QA System\knowledge base\raw
Directory exists: True
DOCUMENT DISCOVERY EXPERIMENT
{
    "root_directory": "d:\\Projects\\Local Mini RAG Document QA System\\knowledge base\\raw",
    "total_files": 6,
    "supported_files": 6,
    "unsupported_files": 0,
    "total_size_bytes": 118951,
    "extension_counts": {
        ".md": 6
    },
    "files": [
        {
            "name": "README_03_Local_Runtime_Engineering.md",
            "relative_path": "knowledge base\\raw\\README_03_Local_Runtime_Engineering.md",
            "extension": ".md",
            "size_bytes": 14492,
            "supported": true
        },
        {
            "name": "README_04_prompt_construction_and_structured_output.md",
            "relative_path": "knowledge base\\raw\\README_04_prompt_construction_and_structured_output.m

## Result Interpretation

### Discovery Statistics

| Metric | Result |
|---|---:|
| Total Files | 6 |
| Supported Files | 6 |
| Unsupported Files | 0 |
| Total Size | 118,951 Bytes |
| Markdown Files | 6 |
| TXT Files | 0 |
| PDF Files | 0 |

### Findings

The document discovery process successfully identified all six files in the configured raw knowledge base directory.

All discovered files use the Markdown extension, which is included in the supported extension allowlist.

The discovery process uses recursive directory traversal and deterministic sorting to produce a reproducible file inventory.

Project-relative paths were generated successfully after correcting the project root resolution.

### Engineering Observations

The experiment confirmed that document discovery should be independent of document extraction.

Discovery identifies candidate files and collects filesystem metadata without reading or interpreting their contents.

The current corpus contains only Markdown files. Additional fixtures will be required to evaluate unsupported extensions, nested directories, empty files, inaccessible files, and file size restrictions.

### Limitations

The experiment evaluated the existing raw knowledge base only.

It did not test malformed files, filesystem permission errors, symbolic links, path traversal attempts, or large-file rejection.

The results describe the current corpus and should not be generalized to arbitrary document collections.

# 03. Document Schema Design

## Objective

Understand how structured document records represent source files and their extracted content.

## Key Concepts

- Document identity
- Source provenance
- File metadata
- Content hashing
- Extraction metadata
- Document records versus page records
- Schema validation

## Document Record

A document record represents an original source file and stores its extracted content alongside its identifying information.

## Page Record

A page record represents an individual page extracted from a document.

Page-level records are particularly important for PDF documents because they preserve page provenance for future source attribution.

## Design Requirements

The schema should:

1. Provide a stable document identifier.
2. Preserve the original source path.
3. Store the original file name and extension.
4. Store the extracted content.
5. Support content hashing for change detection.
6. Preserve file size and ingestion metadata.
7. Support optional page-level information.
8. Validate data types and required fields.

## Expected Outcome

A clear distinction between document-level and page-level data models that can be implemented using Pydantic.

In [5]:
from pathlib import Path
from datetime import datetime, timezone
from hashlib import sha256
import json

PROJECT_ROOT = Path.cwd().parent

sample_file = (
    PROJECT_ROOT / "knowledge base" / "raw" / "README_03_Local_Runtime_Engineering.md"
)

content = sample_file.read_text(encoding="utf-8")
file_bytes = sample_file.read_bytes()

document_record = {
    "document_id": sha256(
        str(sample_file.relative_to(PROJECT_ROOT)).encode("utf-8")
    ).hexdigest()[:16],
    "file_name": sample_file.name,
    "source_path": str(sample_file.relative_to(PROJECT_ROOT)),
    "file_type": sample_file.suffix.lower(),
    "content": content,
    "content_hash": sha256(file_bytes).hexdigest(),
    "file_size": sample_file.stat().st_size,
    "ingested_at": datetime.now(timezone.utc).isoformat(),
    "metadata": {"encoding": "utf-8", "source_type": "local_file"},
}

print("DOCUMENT RECORD")
print("=" * 60)

print(
    json.dumps(
        {key: value for key, value in document_record.items() if key != "content"},
        indent=4,
        ensure_ascii=False,
    )
)

print("\nCONTENT PREVIEW")
print("=" * 60)
print(document_record["content"][:500])

print("\nVALIDATION")
print("=" * 60)

print("Document ID exists:", bool(document_record["document_id"]))
print("Content exists:", bool(document_record["content"]))
print("Content hash length:", len(document_record["content_hash"]))
print("File size matches:", len(file_bytes) == document_record["file_size"])

DOCUMENT RECORD
{
    "document_id": "5824500a14b09ea1",
    "file_name": "README_03_Local_Runtime_Engineering.md",
    "source_path": "knowledge base\\raw\\README_03_Local_Runtime_Engineering.md",
    "file_type": ".md",
    "content_hash": "1aa2321b3340f18c43df43e935cf7634cfca5b9dfd71c903afc39c5d1e5e064e",
    "file_size": 14492,
    "ingested_at": "2026-11-01T21:07:05.368592+00:00",
    "metadata": {
        "encoding": "utf-8",
        "source_type": "local_file"
    }
}

CONTENT PREVIEW
# Local LLM Runtime Engineering

## Overview

Local LLM Runtime Engineering focuses on building a structured runtime integration layer for the Local AI Assistant.

The implementation separates application-level generation requests from the underlying LLM runtime. It provides validated request and response schemas, centralized configuration, an Ollama HTTP adapter, normalized streaming events, runtime error handling, and automated tests.

The goal is to establish a maintainable runtime foundation

V

## Result Interpretation

### Document Record Inspection

The experiment successfully generated a structured document record from the source Markdown file.

The record contains the document identifier, original file name, relative source path, file extension, content hash, file size, ingestion timestamp, and additional metadata.

### Validation Results

| Validation | Result |
|---|---|
| Document ID exists | Passed |
| Extracted content exists | Passed |
| SHA-256 hash length is 64 characters | Passed |
| Recorded file size matches actual file size | Passed |

### Findings

The experiment demonstrated how document identity and content integrity can be represented using separate identifiers.

The document ID was derived from the relative source path, while the content hash was calculated using SHA-256 over the original file bytes.

This distinction allows the system to identify a source document and detect changes to its underlying content independently.

The ingestion timestamp provides a record of when the document entered the ingestion process.

### Important Observation

The file size validation confirmed that the recorded byte count matches the actual source file size.

However, this check verifies size consistency only. It does not establish that the file content is correct, complete, or uncorrupted.

### Limitations

The experiment evaluated a single Markdown document.

It did not test document updates, file relocation, duplicate content, hash collisions, or multiple ingestion runs.

# 04. Pydantic Document Schema

## Objective

Implement validated document and page data models using Pydantic.

## Key Concepts

- Data validation
- Required and optional fields
- Type constraints
- Field validation
- Serialization
- Deserialization
- Schema consistency

## DocumentRecord

Represents an original source document and stores its identity, source information, extracted content, content hash, file size, and ingestion metadata.

## PageRecord

Represents an individual page extracted from a source document.

Each page record maintains a reference to its parent document and preserves its page number and extracted text.

## Design Principles

1. Use explicit data types.
2. Validate required fields.
3. Reject invalid identifiers and hashes.
4. Prevent negative file sizes and page numbers.
5. Use timezone-aware timestamps.
6. Support optional metadata.
7. Keep document-level and page-level information separate.

## Expected Outcome

Two reusable Pydantic models that provide a consistent data contract for the ingestion pipeline.

In [6]:
from datetime import datetime, timezone
from typing import Any

from pydantic import BaseModel, ConfigDict, Field


class DocumentRecord(BaseModel):
    """
    Represents an original source document in the knowledge base.

    The model stores document identity, source information,
    extracted content, integrity information, and ingestion metadata.
    """

    # Configure validation behavior and prevent unexpected fields.
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=False,
    )

    # Stable identifier derived from the document's source identity.
    document_id: str = Field(
        min_length=1,
        description="Unique identifier of the source document.",
    )

    # Original file name, including its extension.
    file_name: str = Field(
        min_length=1,
        description="Original name of the source file.",
    )

    # Path relative to the project or configured knowledge base root.
    source_path: str = Field(
        min_length=1,
        description="Relative path of the original source file.",
    )

    # File extension used to identify the source format.
    file_type: str = Field(
        min_length=1,
        description="Normalized file extension, such as .pdf or .md.",
    )

    # Complete extracted text from the source document.
    content: str = Field(
        description="Extracted textual content of the document.",
    )

    # SHA-256 digest used to detect content changes.
    content_hash: str = Field(
        pattern=r"^[a-fA-F0-9]{64}$",
        description="SHA-256 hash of the source file content.",
    )

    # Original file size measured in bytes.
    file_size: int = Field(
        ge=0,
        description="Original file size in bytes.",
    )

    # Time when the document entered the ingestion pipeline.
    ingested_at: datetime = Field(
        default_factory=lambda: datetime.now(timezone.utc),
        description="Timezone-aware ingestion timestamp.",
    )

    # Flexible metadata for additional source information.
    metadata: dict[str, Any] = Field(
        default_factory=dict,
        description="Additional document metadata.",
    )


class PageRecord(BaseModel):
    """
    Represents an individual page extracted from a document.

    Each page is linked to its parent document to preserve provenance.
    """

    # Apply the same strict handling of unexpected fields.
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=False,
    )

    # Identifier of the parent document.
    document_id: str = Field(
        min_length=1,
        description="Identifier of the parent document.",
    )

    # One-based page number within the original document.
    page_number: int = Field(
        ge=1,
        description="One-based page number.",
    )

    # Text extracted from this specific page.
    content: str = Field(
        description="Extracted textual content of the page.",
    )

    # Optional metadata describing the extraction process.
    metadata: dict[str, Any] = Field(
        default_factory=dict,
        description="Additional page-level metadata.",
    )


print("DocumentRecord fields:")
for field_name, field_info in DocumentRecord.model_fields.items():
    print(f"- {field_name}: {field_info.annotation}")

print("\nPageRecord fields:")
for field_name, field_info in PageRecord.model_fields.items():
    print(f"- {field_name}: {field_info.annotation}")

DocumentRecord fields:
- document_id: <class 'str'>
- file_name: <class 'str'>
- source_path: <class 'str'>
- file_type: <class 'str'>
- content: <class 'str'>
- content_hash: <class 'str'>
- file_size: <class 'int'>
- ingested_at: <class 'datetime.datetime'>
- metadata: dict[str, typing.Any]

PageRecord fields:
- document_id: <class 'str'>
- page_number: <class 'int'>
- content: <class 'str'>
- metadata: dict[str, typing.Any]


In [7]:
# Reuse the document data generated in the previous experiment.
document = DocumentRecord(**document_record)

# Display the validated document without printing its full content.
print("DOCUMENT SCHEMA VALIDATION")
print("=" * 60)

print("Document ID:", document.document_id)
print("File name:", document.file_name)
print("File type:", document.file_type)
print("File size:", document.file_size)
print("Content hash:", document.content_hash)
print("Ingestion timestamp:", document.ingested_at)

print("\nValidation status:", "PASSED")

DOCUMENT SCHEMA VALIDATION
Document ID: 5824500a14b09ea1
File name: README_03_Local_Runtime_Engineering.md
File type: .md
File size: 14492
Content hash: 1aa2321b3340f18c43df43e935cf7634cfca5b9dfd71c903afc39c5d1e5e064e
Ingestion timestamp: 2026-11-01 21:07:05.368592+00:00

Validation status: PASSED


In [8]:
# Create a sample page record linked to the validated document.
page = PageRecord(
    document_id=document.document_id,
    page_number=1,
    content="This is a sample extracted page.",
    metadata={
        "extraction_method": "text_extraction",
        "source_format": document.file_type,
    },
)

# Display the page-level information.
print("PAGE RECORD")
print("=" * 60)

print(page.model_dump_json(indent=4))

PAGE RECORD
{
    "document_id": "5824500a14b09ea1",
    "page_number": 1,
    "content": "This is a sample extracted page.",
    "metadata": {
        "extraction_method": "text_extraction",
        "source_format": ".md"
    }
}


## Result Interpretation

### Schema Inspection

The experiment successfully defined two Pydantic models: DocumentRecord and PageRecord.

DocumentRecord contains nine fields covering document identity, source information, extracted content, integrity information, file size, ingestion timestamp, and additional metadata.

PageRecord contains four fields representing page identity, its parent document, extracted content, and page-level metadata.

### Validation Results

| Validation | Result |
|---|---|
| DocumentRecord model creation | Passed |
| PageRecord model creation | Passed |
| Field type inspection | Passed |
| DocumentRecord instantiation | Passed |
| PageRecord instantiation | Passed |
| PageRecord JSON serialization | Passed |

### Findings

The experiment demonstrated how Pydantic provides a structured data contract for document ingestion.

Document-level and page-level information are represented by separate models, allowing the system to preserve the relationship between an original document and its extracted pages.

The models also define field constraints, including minimum string lengths, non-negative file sizes, positive page numbers, and a 64-character hexadecimal content hash.

The ingestion timestamp uses a timezone-aware UTC default when no explicit timestamp is provided.

### Engineering Observations

The DocumentRecord model represents the source document, while PageRecord represents an individual extracted page.

This separation will support page-level provenance and future source attribution in the Retrieval-Augmented Generation pipeline.

The current implementation uses flexible metadata dictionaries to accommodate additional extraction information without introducing new schema fields for every metadata attribute.

### Limitations

The experiment tested valid sample records only.

It did not evaluate invalid field values, unexpected fields, missing required fields, or complete serialization and deserialization workflows.

Additional validation tests are required before integrating these models into the production ingestion pipeline.

# 05. Testing Schema Validation

## Objective

Evaluate whether the Pydantic document and page schemas correctly reject invalid data and enforce the constraints defined in their field configurations.

## Why Is This Important?

A document ingestion pipeline processes files from different sources, and their extracted data cannot always be assumed to be valid.

Invalid records may contain missing identifiers, malformed hashes, negative file sizes, or incorrect page numbers.

If invalid records enter the knowledge base, they may cause inconsistencies in document storage, retrieval, and source attribution.

Schema validation provides an early boundary that prevents malformed data from propagating through the ingestion pipeline.

## Key Concepts

### 1. Input Validation

Input validation checks whether incoming data satisfies the requirements defined by a data model.

### 2. Field Constraints

Pydantic supports constraints such as:

- `min_length`: Enforces minimum string length.
- `ge`: Requires a numeric value to be greater than or equal to a specified value.
- `pattern`: Validates strings against a regular expression.
- `extra="forbid"`: Rejects fields that are not explicitly defined in the model.

### 3. ValidationError

Pydantic raises a `ValidationError` when input data violates one or more model constraints.

The exception provides structured information about the invalid fields and the reasons for rejection.

### 4. Positive and Negative Testing

- Positive testing verifies that valid data is accepted.
- Negative testing verifies that invalid data is rejected.

Both are necessary to evaluate a data validation system.

## Experimental Procedure

We will perform five controlled negative validation tests:

| Test | Invalid Input | Expected Behavior |
|---|---|---|
| Empty document ID | Empty string | Reject |
| Invalid content hash | Malformed SHA-256 hash | Reject |
| Negative file size | -100 bytes | Reject |
| Zero page number | Page number = 0 | Reject |
| Unexpected field | Undefined model field | Reject |

For each test, we will:

1. Start with a valid baseline record.
2. Modify one field to introduce an invalid value.
3. Attempt to instantiate the corresponding Pydantic model.
4. Capture the validation result.
5. Inspect the error type and affected field.
6. Compare the actual result with the expected behavior.

## Expected Outcome

All five invalid inputs should be rejected by Pydantic.

The experiment should demonstrate that the document and page schemas enforce their defined constraints and provide structured validation errors.

## Engineering Considerations

This experiment evaluates schema-level validation only.

It does not verify whether a source file exists, whether its content is extractable, or whether its metadata is semantically correct.

Those responsibilities belong to later stages of the ingestion pipeline.

In [9]:
from copy import deepcopy

from pydantic import ValidationError


# Create a valid baseline record for controlled validation tests.
valid_document_data = document.model_dump()

# Define invalid inputs to evaluate the schema constraints.
invalid_document_cases = {
    "empty_document_id": {
        **valid_document_data,
        "document_id": "",
    },
    "invalid_content_hash": {
        **valid_document_data,
        "content_hash": "invalid_hash",
    },
    "negative_file_size": {
        **valid_document_data,
        "file_size": -100,
    },
    "unexpected_field": {
        **valid_document_data,
        "unknown_field": "unexpected value",
    },
}

# Store validation outcomes for later analysis.
validation_results = []

# Test each invalid document record independently.
for test_name, test_data in invalid_document_cases.items():
    try:
        # Attempt to create a document using invalid input.
        DocumentRecord(**test_data)

        result = "UNEXPECTEDLY ACCEPTED"
        error_details = None

    except ValidationError as error:
        # Pydantic correctly rejected the invalid record.
        result = "REJECTED"
        error_details = [
            {
                "field": ".".join(str(part) for part in item["loc"]),
                "error": item["type"],
            }
            for item in error.errors()
        ]

    validation_results.append(
        {
            "test": test_name,
            "result": result,
            "errors": error_details,
        }
    )


# Test an invalid page number independently.
try:
    # Page numbers must start at one.
    PageRecord(
        document_id=document.document_id,
        page_number=0,
        content="Invalid page number test.",
    )

    page_result = "UNEXPECTEDLY ACCEPTED"
    page_errors = None

except ValidationError as error:
    # Capture the validation error details.
    page_result = "REJECTED"
    page_errors = [
        {
            "field": ".".join(str(part) for part in item["loc"]),
            "error": item["type"],
        }
        for item in error.errors()
    ]


# Print a structured summary of all validation experiments.
print("=" * 65)
print("PYDANTIC VALIDATION EXPERIMENT")
print("=" * 65)

for result in validation_results:
    print(f"\nTest: {result['test']}")
    print(f"Result: {result['result']}")
    print("Errors:", result["errors"])

print("\nTest: zero_page_number")
print(f"Result: {page_result}")
print("Errors:", page_errors)

# Calculate the overall validation test result.
all_results = [item["result"] for item in validation_results] + [page_result]

passed_tests = sum(result == "REJECTED" for result in all_results)

print("\n" + "=" * 65)
print(f"REJECTED INVALID INPUTS: {passed_tests}/{len(all_results)}")
print("=" * 65)

PYDANTIC VALIDATION EXPERIMENT

Test: empty_document_id
Result: REJECTED
Errors: [{'field': 'document_id', 'error': 'string_too_short'}]

Test: invalid_content_hash
Result: REJECTED
Errors: [{'field': 'content_hash', 'error': 'string_pattern_mismatch'}]

Test: negative_file_size
Result: REJECTED
Errors: [{'field': 'file_size', 'error': 'greater_than_equal'}]

Test: unexpected_field
Result: REJECTED
Errors: [{'field': 'unknown_field', 'error': 'extra_forbidden'}]

Test: zero_page_number
Result: REJECTED
Errors: [{'field': 'page_number', 'error': 'greater_than_equal'}]

REJECTED INVALID INPUTS: 5/5


## Result Interpretation

### Validation Results

The experiment successfully rejected all five invalid input cases.

| Test Case | Error Type | Result |
|---|---|---|
| Empty document ID | string_too_short | Rejected |
| Invalid content hash | string_pattern_mismatch | Rejected |
| Negative file size | greater_than_equal | Rejected |
| Unexpected field | extra_forbidden | Rejected |
| Zero page number | greater_than_equal | Rejected |

Overall Result: 5/5 invalid inputs rejected successfully.

### Findings

The experiment demonstrated that Pydantic enforces the field constraints defined in the DocumentRecord and PageRecord models.

The minimum string length constraint prevented empty document identifiers.

The regular expression constraint rejected malformed SHA-256 hashes.

The numeric constraints prevented negative file sizes and invalid page numbers.

The extra="forbid" configuration prevented unexpected fields from entering the document schema.

Pydantic also returned structured validation errors containing the affected field and the corresponding error type.

### Engineering Implications

Schema validation provides an important data integrity boundary within the ingestion pipeline.

Invalid records can be rejected before they reach document persistence, chunking, embedding, or retrieval stages.

Structured validation errors can also support logging, debugging, and ingestion failure reporting.

### Limitations

The experiment evaluated schema-level constraints using controlled invalid inputs.

It did not evaluate semantic content correctness, source file availability, extraction completeness, or filesystem-level validation.

### Conclusion

The DocumentRecord and PageRecord models successfully enforced all five tested validation constraints.

The schemas are ready for further serialization and deserialization experiments, subject to additional validation requirements identified during implementation.

# 06. Serialization and Deserialization

## Objective

Understand how Pydantic models can be converted into persistent data formats and reconstructed into validated Python objects.

## Why Is This Important?

The ingestion pipeline must store processed documents so they can be reused without repeatedly extracting their original source files.

Document records need to move between application memory, persistent storage, and later processing stages.

Serialization provides a standardized representation for storing and transferring structured data.

Deserialization reconstructs the original data model while applying its validation rules.

## Key Concepts

### 1. Serialization

Serialization converts a Python object into a representation suitable for storage or transmission.

In Pydantic:

- model_dump() returns a Python dictionary.
- model_dump_json() returns a JSON string.

### 2. Deserialization

Deserialization reconstructs a Pydantic model from previously serialized data.

In Pydantic:

- model_validate() validates a Python dictionary.
- model_validate_json() validates and reconstructs a model from a JSON string.

### 3. JSON Compatibility

JSON supports basic data types such as strings, numbers, booleans, arrays, objects, and null values.

Python-specific objects, such as datetime instances, require conversion into JSON-compatible representations.

### 4. Round-Trip Consistency

A round-trip operation serializes an object and then reconstructs it.

The reconstructed object should preserve the original information and satisfy the same schema constraints.

## Experimental Procedure

1. Serialize a DocumentRecord into a Python dictionary.
2. Serialize the same record into a JSON string.
3. Inspect how the datetime field is represented.
4. Reconstruct the model from the JSON string.
5. Compare the original and reconstructed records.
6. Serialize and reconstruct a PageRecord.
7. Verify round-trip consistency.

## Expected Outcome

Both document and page records should survive serialization and deserialization without losing their stored information.

The reconstructed records should pass Pydantic validation and remain equivalent to their original models.

## Engineering Considerations

JSON serialization is useful for document persistence and data exchange.

The experiment evaluates data representation and reconstruction, not database transactions, concurrent writes, or long-term storage reliability.

In [10]:
import json

from pydantic import ValidationError


# ---------------------------------------------------------
# 1. Serialize DocumentRecord into a Python dictionary.
# ---------------------------------------------------------

# Convert the Pydantic model into a standard Python dictionary.
document_dict = document.model_dump()

print("PYTHON DICTIONARY")
print("=" * 60)
print("Data type:", type(document_dict))
print("Ingestion timestamp type:", type(document_dict["ingested_at"]))


# ---------------------------------------------------------
# 2. Serialize DocumentRecord into a JSON string.
# ---------------------------------------------------------

# Convert the model into JSON-compatible text.
document_json = document.model_dump_json(indent=4)

print("\nJSON SERIALIZATION")
print("=" * 60)
print("Data type:", type(document_json))
print("JSON length:", len(document_json))

# Parse JSON to inspect its underlying structure.
parsed_document_json = json.loads(document_json)

print("Serialized timestamp:", parsed_document_json["ingested_at"])


# ---------------------------------------------------------
# 3. Reconstruct DocumentRecord from JSON.
# ---------------------------------------------------------

# Validate the JSON and reconstruct the original Pydantic model.
restored_document = DocumentRecord.model_validate_json(document_json)

print("\nDOCUMENT DESERIALIZATION")
print("=" * 60)
print("Restored model:", type(restored_document))
print("Document ID:", restored_document.document_id)
print("Timestamp type:", type(restored_document.ingested_at))


# ---------------------------------------------------------
# 4. Compare original and reconstructed records.
# ---------------------------------------------------------

# Check whether the reconstructed model preserves all fields.
document_is_equal = document == restored_document

print("\nDOCUMENT ROUND-TRIP VALIDATION")
print("=" * 60)
print("Original equals restored:", document_is_equal)
print("Content preserved:", document.content == restored_document.content)
print("Metadata preserved:", document.metadata == restored_document.metadata)


# ---------------------------------------------------------
# 5. Serialize and reconstruct PageRecord.
# ---------------------------------------------------------

# Convert the page model into JSON.
page_json = page.model_dump_json(indent=4)

# Reconstruct the page model from its JSON representation.
restored_page = PageRecord.model_validate_json(page_json)

print("\nPAGE ROUND-TRIP VALIDATION")
print("=" * 60)
print("Original equals restored:", page == restored_page)
print("Page number:", restored_page.page_number)
print("Parent document ID:", restored_page.document_id)


# ---------------------------------------------------------
# 6. Summarize the experiment.
# ---------------------------------------------------------

round_trip_results = {
    "document_serialization": isinstance(document_json, str),
    "document_deserialization": isinstance(restored_document, DocumentRecord),
    "document_round_trip": document == restored_document,
    "page_serialization": isinstance(page_json, str),
    "page_deserialization": isinstance(restored_page, PageRecord),
    "page_round_trip": page == restored_page,
}

print("\nEXPERIMENT SUMMARY")
print("=" * 60)
print(json.dumps(round_trip_results, indent=4))

print(
    "\nSuccessful checks:",
    sum(round_trip_results.values()),
    "/",
    len(round_trip_results),
)

PYTHON DICTIONARY
Data type: <class 'dict'>
Ingestion timestamp type: <class 'datetime.datetime'>

JSON SERIALIZATION
Data type: <class 'str'>
JSON length: 15343
Serialized timestamp: 2026-11-01T21:07:05.368592Z

DOCUMENT DESERIALIZATION
Restored model: <class '__main__.DocumentRecord'>
Document ID: 5824500a14b09ea1
Timestamp type: <class 'datetime.datetime'>

DOCUMENT ROUND-TRIP VALIDATION
Original equals restored: True
Content preserved: True
Metadata preserved: True

PAGE ROUND-TRIP VALIDATION
Original equals restored: True
Page number: 1
Parent document ID: 5824500a14b09ea1

EXPERIMENT SUMMARY
{
    "document_serialization": true,
    "document_deserialization": true,
    "document_round_trip": true,
    "page_serialization": true,
    "page_deserialization": true,
    "page_round_trip": true
}

Successful checks: 6 / 6


## Result Interpretation

### Serialization Results

The experiment successfully converted DocumentRecord into both a Python dictionary and a JSON string.

The model_dump() method preserved Python-native data types, including the datetime object.

The model_dump_json() method converted the document into a JSON-compatible string representation.

The ingestion timestamp was serialized using an ISO 8601 UTC representation.

### Deserialization Results

The model_validate_json() method successfully reconstructed the DocumentRecord from its serialized JSON representation.

The datetime field was restored as a Python datetime object rather than remaining a string.

The same serialization and deserialization process was successfully performed on PageRecord.

### Round-Trip Validation

| Validation | Result |
|---|---|
| Document serialization | Passed |
| Document deserialization | Passed |
| Document equality | Passed |
| Content preservation | Passed |
| Metadata preservation | Passed |
| Page round-trip | Passed |

Overall Result: 6/6 successful checks.

### Findings

The experiment demonstrated that Pydantic models can be serialized into JSON and reconstructed without losing the tested document or page information.

The reconstructed DocumentRecord was equal to the original model.

The extracted content and metadata were preserved during the round-trip operation.

The datetime field was correctly converted into an ISO 8601 representation during serialization and restored into a Python datetime object during deserialization.

### Engineering Implications

Serialization and deserialization provide the foundation for persistent document storage.

The ingestion pipeline can serialize validated document records before storing them and reconstruct validated models when loading previously processed records.

JSON provides a portable representation that can be exchanged between different application components.

### Important Observation

The model_dump() method returns Python-native values, whereas model_dump_json() returns a JSON string.

These methods serve different purposes and should not be treated as interchangeable.

### Limitations

The experiment evaluated individual document and page records in memory.

It did not test persistent file storage, JSONL record processing, interrupted writes, duplicate records, concurrent access, or recovery from corrupted storage.

### Conclusion

Both DocumentRecord and PageRecord successfully passed the serialization and deserialization experiments.

The models demonstrated complete round-trip consistency across all six tested checks.

# 07. JSONL Persistence

## Objective

Implement a basic persistent storage mechanism for validated document records using the JSON Lines format.

## Why Is This Important?

The previous experiments validated individual document records and demonstrated how to serialize and reconstruct them.

However, the ingestion pipeline must also store multiple processed documents on disk.

Persistent storage allows the system to reuse previously extracted documents without repeating the entire extraction process.

## Key Concepts

### 1. JSON

JSON represents structured data using objects, arrays, strings, numbers, booleans, and null values.

A JSON array can contain multiple document records, but updating the complete file may require rewriting the entire structure.

### 2. JSON Lines

JSONL stores one independent JSON object per line.

Each line represents one complete record and can be processed independently.

### 3. Append Operations

Append mode adds new records to the end of an existing file without replacing its previous contents.

### 4. Record Reconstruction

Each JSONL line can be parsed independently and reconstructed into a validated Pydantic model.

### 5. Persistence

Persistence means storing application data outside the current process memory so it can be retrieved in later executions.

## Experimental Procedure

1. Create a temporary JSONL storage file.
2. Serialize a DocumentRecord into JSON.
3. Append multiple document records.
4. Inspect the resulting file structure.
5. Read the file line by line.
6. Parse each line independently.
7. Reconstruct DocumentRecord instances.
8. Compare reconstructed records with the original records.

## Expected Outcome

The experiment should demonstrate that multiple document records can be stored as independent JSONL entries and reconstructed without losing their information.

## Engineering Considerations

This experiment uses a local JSONL file as a simple persistence layer.

It does not implement transactional writes, concurrent access control, atomic file replacement, or database indexing.

These concerns will be considered when designing the production ingestion pipeline.

In [ ]:
import json
from pathlib import Path

from pydantic import ValidationError


# ---------------------------------------------------------
# 1. Prepare the experimental storage directory.
# ---------------------------------------------------------

# Keep experimental persistence files separate from production data.
EXPERIMENT_DIR = PROJECT_ROOT / "data" / "experiments" / "day1"

EXPERIMENT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Define the JSONL file used by this experiment.
JSONL_PATH = EXPERIMENT_DIR / "documents_experiment.jsonl"

# Remove any previous experimental file to ensure reproducibility.
if JSONL_PATH.exists():
    JSONL_PATH.unlink()


# ---------------------------------------------------------
# 2. Prepare multiple document records.
# ---------------------------------------------------------

# Use the existing validated document as the first record.
first_document = document

# Create a second controlled document record.
second_document_data = document.model_dump()

second_document_data.update(
    {
        "document_id": "experimental_doc_002",
        "file_name": "sample_experiment.md",
        "source_path": "data/experiments/sample_experiment.md",
        "content": "This is a second experimental document.",
        "content_hash": "a" * 64,
        "file_size": 42,
    }
)

second_document = DocumentRecord(**second_document_data)

# Keep the records together for sequential persistence.
documents_to_store = [
    first_document,
    second_document,
]


# ---------------------------------------------------------
# 3. Write document records using JSONL format.
# ---------------------------------------------------------

# Open the file in append mode and write one JSON object per line.
with JSONL_PATH.open("a", encoding="utf-8") as file:
    for record in documents_to_store:
        # Serialize each Pydantic model into a JSON string.
        json_line = record.model_dump_json()

        # Write one independent record followed by a newline.
        file.write(json_line + "\n")


# ---------------------------------------------------------
# 4. Inspect the persisted file.
# ---------------------------------------------------------

# Read the raw JSONL file as text.
raw_content = JSONL_PATH.read_text(encoding="utf-8")

# Count non-empty lines to determine the number of stored records.
stored_lines = [line for line in raw_content.splitlines() if line.strip()]

print("JSONL STORAGE INSPECTION")
print("=" * 65)
print("Storage path:", JSONL_PATH)
print("File exists:", JSONL_PATH.exists())
print("Stored records:", len(stored_lines))
print("File size:", JSONL_PATH.stat().st_size, "bytes")

print("\nRAW JSONL CONTENT")
print("=" * 65)

for line_number, line in enumerate(stored_lines, start=1):
    print(f"Line {line_number}: {line[:180]}...")


# ---------------------------------------------------------
# 5. Load and reconstruct document records.
# ---------------------------------------------------------

# Store reconstructed models for subsequent comparison.
restored_documents = []

# Process each JSONL line independently.
with JSONL_PATH.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        # Skip empty lines instead of attempting to parse them.
        if not line.strip():
            continue

        try:
            # Validate the JSON line and reconstruct its Pydantic model.
            restored_record = DocumentRecord.model_validate_json(line)

            restored_documents.append(restored_record)

        except ValidationError as error:
            # Report invalid records with their corresponding line numbers.
            print(f"Validation failed at line {line_number}:", error)


# ---------------------------------------------------------
# 6. Verify persistence and reconstruction.
# ---------------------------------------------------------

# Compare the reconstructed records with their original models.
round_trip_checks = [
    original == restored
    for original, restored in zip(documents_to_store, restored_documents)
]

print("\nPERSISTENCE VALIDATION")
print("=" * 65)
print("Original records:", len(documents_to_store))
print("Restored records:", len(restored_documents))
print("All records restored:", len(restored_documents) == len(documents_to_store))
print("Record comparisons:", round_trip_checks)
print("All records preserved:", all(round_trip_checks))


# ---------------------------------------------------------
# 7. Summarize the experiment.
# ---------------------------------------------------------

persistence_summary = {
    "file_created": JSONL_PATH.exists(),
    "stored_records": len(stored_lines),
    "restored_records": len(restored_documents),
    "record_count_matches": len(stored_lines) == len(restored_documents),
    "all_records_preserved": (
        len(round_trip_checks) == len(documents_to_store) and all(round_trip_checks)
    ),
}

print("\nEXPERIMENT SUMMARY")
print("=" * 65)
print(json.dumps(persistence_summary, indent=4))

JSONL STORAGE INSPECTION
Storage path: d:\Projects\Local Mini RAG Document QA System\data\experiments\day1\documents_experiment.jsonl
File exists: True
Stored records: 2
File size: 15664 bytes

RAW JSONL CONTENT
Line 1: {"document_id":"5824500a14b09ea1","file_name":"README_03_Local_Runtime_Engineering.md","source_path":"knowledge base\\raw\\README_03_Local_Runtime_Engineering.md","file_type":".md"...
Line 2: {"document_id":"experimental_doc_002","file_name":"sample_experiment.md","source_path":"data/experiments/sample_experiment.md","file_type":".md","content":"This is a second experim...

PERSISTENCE VALIDATION
Original records: 2
Restored records: 2
All records restored: True
Record comparisons: [True, True]
All records preserved: True

EXPERIMENT SUMMARY
{
    "file_created": true,
    "stored_records": 2,
    "restored_records": 2,
    "record_count_matches": true,
    "all_records_preserved": true
}


## Result Interpretation

### Storage Results

The experiment successfully created a JSONL file and persisted two DocumentRecord instances.

| Metric | Result |
|---|---:|
| File Created | True |
| Stored Records | 2 |
| File Size | 15,664 Bytes |
| Restored Records | 2 |
| Record Count Matches | True |
| All Records Preserved | True |

### Reconstruction Results

Both JSONL records were successfully read and reconstructed into validated DocumentRecord instances.

The equality comparisons returned:

[True, True]

This confirms that both reconstructed records matched their original Pydantic models.

### Findings

The experiment demonstrated that JSONL can store multiple independent document records using one JSON object per line.

Each record was serialized independently and appended to the storage file.

During loading, each line was parsed and validated independently using DocumentRecord.model_validate_json().

Both records survived the persistence round trip without losing their tested fields or metadata.

### Engineering Implications

JSONL provides a simple persistence format for the document ingestion pipeline.

Its line-oriented structure supports sequential processing and allows individual records to be loaded without parsing an entire JSON array.

This approach is suitable for the initial document storage layer of the project.

### Important Observation

Append mode does not provide duplicate prevention.

Running the same ingestion operation multiple times may append identical document records repeatedly.

A separate deduplication mechanism is required to make ingestion idempotent.

### Limitations

The experiment used two controlled records and a local filesystem.

It did not evaluate concurrent writes, interrupted operations, corrupted JSONL lines, atomic updates, or large-scale storage performance.

### Conclusion

Both document records were successfully persisted and reconstructed.

The experiment established a working foundation for JSONL-based document storage, while highlighting the need for duplicate detection and idempotent ingestion.

# 08. Deduplication and Idempotency

## Objective

Understand how document identity and content hashing can be used to detect duplicate documents and identify changes during repeated ingestion runs.

## Why Is This Important?

An ingestion pipeline may execute repeatedly as new documents are added or existing documents are updated.

Without duplicate detection, repeated executions may append identical records to persistent storage.

This can increase storage consumption and introduce duplicate content into subsequent chunking, embedding, and retrieval stages.

## Key Concepts

### 1. Document Identity

Document identity represents the logical source document.

In this experiment, the document ID is derived from its relative source path.

### 2. Content Hashing

A content hash represents the contents of a document.

SHA-256 provides a deterministic digest for identical byte sequences.

### 3. Deduplication

Deduplication identifies records that already exist according to a defined identity or content policy.

### 4. Idempotency

An ingestion operation is idempotent when repeating it with unchanged inputs does not create additional unintended records.

### 5. Document Updates

When a document retains the same source path but its content changes, its document ID remains stable while its content hash changes.

This distinction allows the ingestion system to identify updated documents.

## Experimental Procedure

1. Load the existing JSONL records.
2. Create a lookup structure using document IDs.
3. Simulate a repeated ingestion of an unchanged document.
4. Verify that the duplicate is skipped.
5. Simulate a document update with the same document ID and a different content hash.
6. Identify the update without treating it as a new document.
7. Simulate identical content stored under a different source path.
8. Compare document identity and content hashes.

## Expected Outcome

The experiment should distinguish between:

- New documents.
- Unchanged documents.
- Updated documents.
- Documents with identical content but different source paths.

Repeated ingestion of unchanged documents should not increase the number of stored records.

## Engineering Considerations

Document identity and content equality represent different concepts.

A document can change without changing its identity, and two different documents can contain identical content.

The deduplication policy must preserve this distinction rather than relying on a single identifier.

In [ ]:
from copy import deepcopy
from hashlib import sha256
from pathlib import Path
import json


# ---------------------------------------------------------
# 1. Load existing records from JSONL storage.
# ---------------------------------------------------------

# Read every valid JSONL line and reconstruct its document model.
existing_documents = []

with JSONL_PATH.open("r", encoding="utf-8") as file:
    for line in file:
        # Ignore empty lines.
        if not line.strip():
            continue

        # Reconstruct and validate each stored document.
        existing_documents.append(DocumentRecord.model_validate_json(line))


# ---------------------------------------------------------
# 2. Build a document identity lookup.
# ---------------------------------------------------------

# Use document IDs as keys for efficient identity checks.
documents_by_id = {record.document_id: record for record in existing_documents}

print("EXISTING STORAGE")
print("=" * 65)
print("Existing records:", len(existing_documents))
print("Unique document IDs:", len(documents_by_id))


# ---------------------------------------------------------
# 3. Simulate repeated ingestion of an unchanged document.
# ---------------------------------------------------------

# Reuse the first stored document without modifying its content.
repeated_document = deepcopy(existing_documents[0])

# Check whether the document already exists.
existing_record = documents_by_id.get(repeated_document.document_id)

if existing_record is None:
    repeated_result = "NEW_DOCUMENT"

elif existing_record.content_hash == repeated_document.content_hash:
    # Same identity and same content indicate an unchanged document.
    repeated_result = "SKIPPED_UNCHANGED"

else:
    # Same identity but different content indicates an update.
    repeated_result = "UPDATED_DOCUMENT"


print("\nREPEATED INGESTION")
print("=" * 65)
print("Document ID:", repeated_document.document_id)
print(
    "Content hash matches:",
    (existing_record.content_hash == repeated_document.content_hash),
)
print("Ingestion decision:", repeated_result)


# ---------------------------------------------------------
# 4. Simulate an updated document.
# ---------------------------------------------------------

# Copy the existing document data to preserve its identity.
updated_document_data = existing_documents[0].model_dump()

# Modify the extracted content.
updated_content = (
    updated_document_data["content"] + "\n\nThis document has been updated."
)

# Recalculate the content hash using the updated content bytes.
updated_content_hash = sha256(updated_content.encode("utf-8")).hexdigest()

# Create a new record with the same identity and updated content.
updated_document_data.update(
    {
        "content": updated_content,
        "content_hash": updated_content_hash,
        "file_size": len(updated_content.encode("utf-8")),
    }
)

updated_document = DocumentRecord(**updated_document_data)

# Compare the original and updated records.
same_identity = existing_documents[0].document_id == updated_document.document_id

same_content = existing_documents[0].content_hash == updated_document.content_hash

if same_identity and not same_content:
    update_result = "UPDATED_DOCUMENT"
else:
    update_result = "UNEXPECTED_RESULT"


print("\nDOCUMENT UPDATE")
print("=" * 65)
print("Same document ID:", same_identity)
print("Same content hash:", same_content)
print("Ingestion decision:", update_result)


# ---------------------------------------------------------
# 5. Simulate identical content under a different path.
# ---------------------------------------------------------

# Copy the original record while preserving its content and hash.
relocated_document_data = existing_documents[0].model_dump()

# Change the source identity and file name.
relocated_document_data.update(
    {
        "document_id": "experimental_doc_003",
        "file_name": "relocated_document.md",
        "source_path": "another_folder/relocated_document.md",
    }
)

relocated_document = DocumentRecord(**relocated_document_data)

# Compare identity and content independently.
different_identity = existing_documents[0].document_id != relocated_document.document_id

identical_content = (
    existing_documents[0].content_hash == relocated_document.content_hash
)

print("\nIDENTICAL CONTENT, DIFFERENT SOURCE")
print("=" * 65)
print("Different document ID:", different_identity)
print("Identical content hash:", identical_content)


# ---------------------------------------------------------
# 6. Summarize the deduplication experiment.
# ---------------------------------------------------------

deduplication_summary = {
    "existing_record_count": len(existing_documents),
    "repeated_document_decision": repeated_result,
    "updated_document_decision": update_result,
    "same_identity_after_update": same_identity,
    "content_changed_after_update": not same_content,
    "different_identity_for_new_path": different_identity,
    "identical_content_for_new_path": identical_content,
}

print("\nEXPERIMENT SUMMARY")
print("=" * 65)
print(json.dumps(deduplication_summary, indent=4))

EXISTING STORAGE
Existing records: 2
Unique document IDs: 2

REPEATED INGESTION
Document ID: 5824500a14b09ea1
Content hash matches: True
Ingestion decision: SKIPPED_UNCHANGED

DOCUMENT UPDATE
Same document ID: True
Same content hash: False
Ingestion decision: UPDATED_DOCUMENT

IDENTICAL CONTENT, DIFFERENT SOURCE
Different document ID: True
Identical content hash: True

EXPERIMENT SUMMARY
{
    "existing_record_count": 2,
    "repeated_document_decision": "SKIPPED_UNCHANGED",
    "updated_document_decision": "UPDATED_DOCUMENT",
    "same_identity_after_update": true,
    "content_changed_after_update": true,
    "different_identity_for_new_path": true,
    "identical_content_for_new_path": true
}


## Result Interpretation

### 1. Repeated Ingestion

- Existing records: 2.
- Unique document IDs: 2.
- The repeated document had the same document ID and content hash as the stored record.
- The ingestion decision was `SKIPPED_UNCHANGED`.

This demonstrates that unchanged documents can be recognized and skipped instead of being processed and stored again.

### 2. Document Update

- The updated document retained its original document ID.
- Its content hash changed.
- The ingestion decision was `UPDATED_DOCUMENT`.

This demonstrates how the system can distinguish an updated source document from a newly discovered document.

### 3. Identical Content From Different Sources

- The relocated document received a different document ID.
- Its content hash remained identical.

This demonstrates that document identity and content duplication are separate concepts. Different source files may contain identical text while remaining independently traceable.

### 4. Engineering Considerations

The experiment used simulated ingestion decisions and did not modify the persistent JSONL storage.

A consistent hashing strategy is required before production implementation:

- `source_hash` should be calculated from the original file bytes.
- `content_hash` should be calculated from a consistently encoded and normalized representation of extracted text.

Separating these hashes allows the pipeline to detect source-file changes independently from extracted-text duplication.

### Finding

The experiment successfully demonstrated three ingestion decisions:

1. `SKIPPED_UNCHANGED`: The same source document has not changed.
2. `UPDATED_DOCUMENT`: The same source document has changed.
3. Different document identity with identical content: Two distinct sources contain equivalent content.

These behaviors establish the foundation for an idempotent ingestion pipeline.

**Limitation:** This experiment validates the decision logic only. It does not test concurrent ingestion, persistent record replacement, transactional updates, or duplicate handling across multiple ingestion runs.

# 09. Production Data Model Design

## Objective

Design the core Pydantic models that define the data contracts for the document ingestion system.

## Why This Matters

A reliable ingestion pipeline requires consistent data structures shared across document discovery, extraction, validation, persistence, and reporting.

The data models will define:

- Document identity and source information.
- Original file and extracted content hashes.
- Document metadata and ingestion timestamps.
- Page-level extraction records.
- Explicit ingestion processing statuses.

## Core Models

### 1. DocumentRecord

Represents a source document and its extracted text.

### 2. PageRecord

Represents an individual extracted page while preserving its relationship with the parent document.

### 3. IngestionResult

Represents the outcome of processing a document, including its status and optional error information.

### 4. IngestionStatus

Defines the supported ingestion outcomes:

- NEW_DOCUMENT
- UPDATED_DOCUMENT
- SKIPPED_UNCHANGED
- DUPLICATE_CONTENT
- FAILED

## Design Requirements

- Use Pydantic for validation.
- Reject unexpected fields.
- Preserve Unicode content.
- Use timezone-aware UTC timestamps.
- Validate SHA-256 hashes.
- Keep source identity separate from content identity.
- Use explicit status values.
- Avoid coupling the models to FastAPI or Ollama.

## Expected Outcome

A small, reusable data-model layer that can be imported by every ingestion component without introducing dependencies on the application runtime.

## Result Interpretation

### Test Execution

The ingestion data models were implemented using Pydantic and tested with pytest.

Test results:

- Total tests: 12
- Passed: 12
- Failed: 0
- Execution time: 0.24 seconds

### Validation Results

The tests verified:

1. Successful creation of DocumentRecord.
2. Automatic generation of timezone-aware UTC timestamps.
3. Rejection of invalid SHA-256 hash values.
4. Rejection of empty document identifiers and filenames.
5. Rejection of negative file sizes.
6. Rejection of unexpected fields.
7. Successful creation of PageRecord.
8. Rejection of invalid page numbers.
9. Successful creation of IngestionResult.
10. Correct definition of all supported ingestion statuses.

### Finding

The initial production data-model implementation passed all 12 tests.

The models establish validated data contracts for document records, page-level extraction records, and ingestion outcomes.

### Limitations

These tests validate model construction and field constraints only.

They do not yet verify:

- Hash calculation against actual source files.
- Referential consistency between documents and pages.
- Ingestion status transitions.
- Persistent storage behavior.
- End-to-end ingestion operations.

# 10. Document Discovery and Source Validation

## Objective

Understand and implement the document discovery stage responsible for identifying valid source files before text extraction.

## Why Document Discovery Matters

An ingestion pipeline must identify which files are eligible for processing before attempting to extract their content.

Discovery separates file-system inspection and source validation from text extraction and document processing.

## Supported Formats

The initial implementation supports:

- Plain text files (.txt)
- Markdown files (.md)
- PDF documents (.pdf)

## Validation Rules

The discovery process evaluates:

1. File extension.
2. File existence and accessibility.
3. File size.
4. Empty files.
5. Symbolic links.
6. Resolved path containment.
7. Deterministic file ordering.

## Discovery Architecture

The discovery module produces three logical outputs:

- DiscoveredFile: Information about a valid source file.
- DiscoveryIssue: Information about a skipped file and its reason.
- DiscoveryReport: A summary of the complete discovery operation.

## Engineering Considerations

Discovery must not extract text, generate embeddings, or communicate with the LLM runtime.

Its responsibility is limited to identifying and validating source files.

## Expected Outcome

A deterministic and secure discovery component that provides validated source-file candidates to the extraction pipeline.

## Result Interpretation

### Test Execution

The document discovery module and production data models were tested using pytest.

Test results:

| Component | Total | Passed | Failed |
|---|---:|---:|---:|
| Data Models | 12 | 12 | 0 |
| Document Discovery | 12 | 12 | 0 |
| Total | 24 | 24 | 0 |

The complete ingestion test suite passed in approximately 0.3 seconds.

### Discovery Validation Results

The discovery tests verified:

1. Discovery of supported TXT, Markdown, and PDF extensions.
2. Rejection of unsupported extensions.
3. Rejection of empty files.
4. Enforcement of maximum file-size limits.
5. Recursive directory discovery.
6. Non-recursive directory discovery.
7. Deterministic file ordering.
8. Handling of missing directories.
9. Rejection of file paths used as source directories.
10. Validation of maximum file-size configuration.
11. Support for custom file extensions.
12. Rejection of hidden files.

### Finding

The initial document discovery implementation passed all 12 dedicated tests.

Together with the 12 data-model tests, the ingestion foundation currently has 24 passing tests.

The discovery component now provides deterministic source-file candidates and structured exclusion reasons without performing text extraction or LLM inference.

### Limitations

The current tests do not establish complete protection against:

- Concurrent filesystem modifications.
- All symbolic-link and path traversal scenarios.
- Permission changes during file processing.
- Malformed PDF documents.
- Extraction failures.

These concerns will be addressed in the relevant implementation and testing stages.

# 11. Real Knowledge Base Discovery Experiment

## Objective

Evaluate the production discovery module against the project's actual raw knowledge base.

## Experimental Setup

The experiment uses the existing `knowledge base/raw` directory as the source corpus.

No files are copied, modified, or deleted.

## Measurements

The experiment records:

- Total discovered files.
- Total skipped files.
- Supported file extensions.
- Total discovered file size.
- Average file size.
- File-size distribution.
- Exclusion reasons.
- Relative source paths.

## Expected Outcome

A reproducible inventory of the actual source corpus, based on the production discovery implementation rather than manually generated fixtures.

## Limitations

This experiment evaluates file discovery and metadata only.

It does not evaluate extraction quality, PDF readability, OCR requirements, or semantic content.

In [15]:
import sys
from collections import Counter
from pathlib import Path

# Ensure the project root is on sys.path so the 'app' package resolves.
_PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from app.ingestion.discovery import discover_documents


# Resolve the project root independently of the notebook location.
CURRENT_DIR = Path.cwd()

PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

# Use the existing raw knowledge base without duplicating its files.
RAW_KNOWLEDGE_BASE = PROJECT_ROOT / "knowledge base" / "raw"

# Run the actual production discovery implementation.
discovery_report = discover_documents(
    RAW_KNOWLEDGE_BASE,
    max_file_size_bytes=25 * 1024 * 1024,
    recursive=True,
)

discovered_files = discovery_report.discovered_files
skipped_files = discovery_report.skipped_files

# Calculate file statistics.
extension_counts = Counter(item.extension for item in discovered_files)

skip_reason_counts = Counter(item.reason for item in skipped_files)

total_size = sum(item.file_size for item in discovered_files)

average_size = total_size / len(discovered_files) if discovered_files else 0

print("=" * 75)
print("REAL KNOWLEDGE BASE DISCOVERY")
print("=" * 75)

print(f"Source directory: {RAW_KNOWLEDGE_BASE}")
print(f"Discovered files: {discovery_report.discovered_count}")
print(f"Skipped files: {discovery_report.skipped_count}")

print("\nFILE EXTENSIONS")
print("-" * 75)

for extension, count in sorted(extension_counts.items()):
    print(f"{extension}: {count}")

print("\nFILE SIZE STATISTICS")
print("-" * 75)

print(f"Total size: {total_size:,} bytes")
print(f"Total size: {total_size / 1024:.2f} KB")
print(f"Average size: {average_size:,.2f} bytes")

print("\nDISCOVERED FILES")
print("-" * 75)

for item in discovered_files:
    print(f"{item.relative_path} | {item.file_size:,} bytes")

print("\nSKIPPED FILES")
print("-" * 75)

if skipped_files:
    for item in skipped_files:
        print(f"{item.path} | Reason: {item.reason}")
else:
    print("No files were skipped.")

print("\nSKIP REASON SUMMARY")
print("-" * 75)

for reason, count in sorted(skip_reason_counts.items()):
    print(f"{reason}: {count}")

print("\nEXPERIMENT COMPLETED")

REAL KNOWLEDGE BASE DISCOVERY
Source directory: d:\Projects\Local Mini RAG Document QA System\knowledge base\raw
Discovered files: 6
Skipped files: 0

FILE EXTENSIONS
---------------------------------------------------------------------------
.md: 6

FILE SIZE STATISTICS
---------------------------------------------------------------------------
Total size: 118,951 bytes
Total size: 116.16 KB
Average size: 19,825.17 bytes

DISCOVERED FILES
---------------------------------------------------------------------------
README_03_Local_Runtime_Engineering.md | 14,492 bytes
README_04_prompt_construction_and_structured_output.md | 9,608 bytes
README_05_end_to_end_local_ai_assistant.md | 32,510 bytes
README_06_prompt_regression_testing.md | 23,258 bytes
README_07_hadota_web_chat.md | 9,503 bytes
README_08_HADOTA_Local_AI_Assistant.md | 29,580 bytes

SKIPPED FILES
---------------------------------------------------------------------------
No files were skipped.

SKIP REASON SUMMARY
-------------

## Result Interpretation

The production document discovery function successfully scanned the project's actual raw knowledge base.

### Observed Results

* Discovered files: 6
* Skipped files: 0
* Supported file types found: Markdown (`.md`) only
* Total corpus size: 118,951 bytes (116.16 KB)
* Average file size: 19,825.17 bytes

All six discovered files passed the current discovery filters, including extension validation, file-size validation, and hidden-file filtering.

### Key Findings

1. The discovery function correctly resolved and scanned the actual project knowledge base directory.
2. All six Markdown documents were discovered successfully.
3. No files were skipped under the current validation rules.
4. The current corpus contains Markdown documents only; TXT and PDF extraction still require separate controlled experiments.
5. The corpus is small enough to support reproducible ingestion experiments and manual extraction-quality inspection.

### Limitations

* Successful discovery does not guarantee successful text extraction.
* The experiment does not evaluate Markdown content quality or structural preservation.
* The corpus does not currently provide real TXT or PDF files for testing those extractors.
* File discovery does not perform OCR, semantic analysis, or document chunking.

## Finding

The document discovery component successfully identified all six files in the real knowledge base without skipping any files.

The discovery results provide a reliable input for the next stage: implementing and testing format-specific document extractors.

# 12. Implement Document Extractors

## Objective

Implement format-specific extraction components that convert supported source documents into structured text extraction results.

The extraction layer is responsible for reading document content while preserving its essential structure and metadata.

It must remain independent of document discovery, ingestion orchestration, persistence, and retrieval.

## Supported Document Formats

| Format   | Extraction Method       | Output Representation                                |
| -------- | ----------------------- | ---------------------------------------------------- |
| TXT      | UTF-8 text decoding     | Single logical page                                  |
| Markdown | UTF-8 text decoding     | Single logical page with original Markdown structure |
| PDF      | PyMuPDF text extraction | Separate extracted page for each PDF page            |

## Extraction Architecture

The extraction module introduces three main data structures:

### ExtractedPage

Represents the extracted content of one logical document page.

Attributes:

* `page_number`: One-based page number.
* `content`: Extracted text.
* `metadata`: Page-level extraction metadata.

### ExtractionResult

Represents the complete extraction output for a source document.

Attributes:

* `source_path`: Original source file path.
* `file_type`: Normalized file extension.
* `pages`: Collection of extracted pages.
* `total_pages`: Number of extracted logical pages.
* `total_characters`: Total number of extracted characters.
* `metadata`: Document-level extraction metadata.

### ExtractionError

A dedicated exception hierarchy used to report extraction failures and unsupported document formats.

## Extraction Strategies

### TXT Extraction

* Read files using UTF-8 encoding.
* Support UTF-8 Byte Order Mark (BOM).
* Preserve original text and line breaks.
* Reject invalid UTF-8 input instead of silently replacing characters.

### Markdown Extraction

* Read files using UTF-8 encoding.
* Preserve headings, lists, tables, and Markdown syntax.
* Avoid converting Markdown into plain text.
* Retain the original document structure for downstream processing.

### PDF Extraction

* Use PyMuPDF (`fitz`) to open PDF documents.
* Extract text independently from each page.
* Preserve one-based page numbering.
* Use sorted text extraction to improve reading order.
* Reject invalid and password-protected documents.
* Preserve PDF metadata when available.

## Unicode and Error Handling

The extraction layer must preserve Unicode characters, including Arabic text.

It must also handle:

* Missing source files.
* Unsupported extensions.
* Invalid text encoding.
* Invalid PDF documents.
* Password-protected PDFs.
* File reading errors.

Extraction failures must be explicit and must not silently produce corrupted text.

## Extraction Quality Considerations

The initial implementation focuses on text-based documents.

The following capabilities are intentionally outside the current scope:

* Optical Character Recognition (OCR).
* Scanned PDF processing.
* Advanced table reconstruction.
* Multi-column layout reconstruction.
* Semantic text cleaning.
* Document chunking.

These capabilities can be evaluated in later stages when required.

## Automated Testing Strategy

The extraction components are evaluated using controlled test fixtures.

The test suite covers:

1. TXT content extraction.
2. Markdown structure preservation.
3. Arabic Unicode preservation.
4. UTF-8 BOM handling.
5. Invalid encoding rejection.
6. Unsupported format rejection.
7. PDF page boundary preservation.
8. PDF character count consistency.
9. Invalid PDF rejection.
10. Password-protected PDF rejection.
11. Document extraction dispatching.
12. Missing source file handling.

## Implementation Outcome

The extraction layer provides a consistent interface for converting TXT, Markdown, and PDF files into structured extraction results.

This abstraction allows the future ingestion pipeline to process multiple document formats without embedding format-specific extraction logic inside the pipeline itself.

## Result Interpretation

The format-specific extraction components were implemented and tested successfully.

### Test Results

| Component                   | Tests Passed | Tests Failed |
| --------------------------- | -----------: | -----------: |
| TXT and Markdown Extraction |            8 |            0 |
| PDF Extraction              |            5 |            0 |
| Document Discovery          |           12 |            0 |
| Ingestion Models            |           12 |            0 |
| **Total**                   |       **37** |        **0** |

The extraction-specific test suite completed in 0.54 seconds.

The complete ingestion test suite completed in 0.74 seconds.

### Key Findings

1. TXT extraction correctly reads UTF-8 content and handles UTF-8 BOM.
2. Markdown extraction preserves the original document structure.
3. Arabic Unicode content is preserved.
4. Invalid UTF-8 input produces an explicit extraction error.
5. PDF extraction preserves individual page boundaries and page numbering.
6. Invalid and password-protected PDFs are rejected.
7. The document dispatcher selects the appropriate extractor based on the file extension.
8. All existing discovery and model tests continue to pass.

### Limitations

* PDF extraction has only been tested with controlled, text-based PDF fixtures.
* Scanned documents and OCR are not supported.
* Complex tables, multi-column layouts, and unusual PDF encodings have not been evaluated.
* Passing unit tests does not establish extraction quality across arbitrary real-world documents.

## Finding

The initial extraction implementation passed all 13 dedicated extraction tests and all 37 ingestion tests.

The TXT, Markdown, and PDF extraction interfaces are now ready for evaluation against the actual project knowledge base.


# 13. Real Knowledge Base Extraction Experiment

## Objective

Evaluate the implemented extraction components against the actual project knowledge base.

The experiment will use the existing document discovery component to identify supported source files and the extraction dispatcher to process each discovered document.

## Evaluation Metrics

* Number of discovered documents.
* Number of successfully extracted documents.
* Number of failed extractions.
* Total logical pages extracted.
* Total extracted characters.
* Number of documents with empty extracted content.
* Total extraction execution time.
* Original source size and extracted text size.

## Experimental Constraints

* Use the existing raw knowledge base without modifying source files.
* Do not introduce OCR.
* Preserve the original Markdown structure.
* Keep extraction results in memory during this experiment.
* Do not write to the production document store.
* Record extraction failures without stopping the entire experiment.

## Expected Outcome

Determine whether the implemented extractors can successfully process the actual project knowledge base and provide measurable extraction statistics.


In [ ]:
import time
from collections import Counter
from pathlib import Path

from app.ingestion.discovery import discover_documents
from app.ingestion.extractors import extract_document, ExtractionError


# Resolve the project root independently of the notebook working directory.
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

RAW_KNOWLEDGE_BASE = PROJECT_ROOT / "knowledge base" / "raw"


# Discover supported source documents.
discovery_report = discover_documents(
    RAW_KNOWLEDGE_BASE,
    recursive=True,
)

discovered_files = discovery_report.discovered_files

# Store extraction results and failures separately.
extraction_results = []
extraction_failures = []

empty_documents = []
total_source_bytes = 0

# Measure extraction execution time.
start_time = time.perf_counter()

for discovered_file in discovered_files:
    total_source_bytes += discovered_file.file_size

    try:
        result = extract_document(discovered_file.path)
        extraction_results.append(result)

        if result.total_characters == 0:
            empty_documents.append(discovered_file.relative_path)

    except Exception as exc:
        extraction_failures.append(
            {
                "source_path": discovered_file.relative_path,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

elapsed_seconds = time.perf_counter() - start_time


# Calculate aggregate extraction metrics.
successful_count = len(extraction_results)
failed_count = len(extraction_failures)

total_pages = sum(result.total_pages for result in extraction_results)

total_characters = sum(result.total_characters for result in extraction_results)

extension_counts = Counter(result.file_type for result in extraction_results)

extracted_text_bytes = sum(
    len(page.content.encode("utf-8"))
    for result in extraction_results
    for page in result.pages
)


# Display the experiment report.
print("=" * 75)
print("REAL KNOWLEDGE BASE EXTRACTION REPORT")
print("=" * 75)

print(f"Discovered documents: {len(discovered_files)}")
print(f"Successfully extracted: {successful_count}")
print(f"Failed extractions: {failed_count}")
print(f"Empty documents: {len(empty_documents)}")

print("\nEXTRACTION STATISTICS")
print("-" * 75)

print(f"Total logical pages: {total_pages}")
print(f"Total extracted characters: {total_characters:,}")
print(f"Total source size: {total_source_bytes:,} bytes")
print(f"Extracted text size: {extracted_text_bytes:,} bytes")
print(f"Execution time: {elapsed_seconds:.6f} seconds")

print("\nEXTRACTED DOCUMENTS")
print("-" * 75)

for result in extraction_results:
    print(
        f"{Path(result.source_path).name} | "
        f"Pages: {result.total_pages} | "
        f"Characters: {result.total_characters:,}"
    )

print("\nFILE TYPE SUMMARY")
print("-" * 75)

for extension, count in sorted(extension_counts.items()):
    print(f"{extension}: {count}")

print("\nEMPTY DOCUMENTS")
print("-" * 75)

if empty_documents:
    for document in empty_documents:
        print(document)
else:
    print("No empty documents.")

print("\nEXTRACTION FAILURES")
print("-" * 75)

if extraction_failures:
    for failure in extraction_failures:
        print(failure)
else:
    print("No extraction failures.")

print("\nEXPERIMENT COMPLETED")

REAL KNOWLEDGE BASE EXTRACTION REPORT
Discovered documents: 6
Successfully extracted: 6
Failed extractions: 0
Empty documents: 0

EXTRACTION STATISTICS
---------------------------------------------------------------------------
Total logical pages: 6
Total extracted characters: 118,204
Total source size: 118,951 bytes
Extracted text size: 118,383 bytes
Execution time: 0.081509 seconds

EXTRACTED DOCUMENTS
---------------------------------------------------------------------------
README_03_Local_Runtime_Engineering.md | Pages: 1 | Characters: 14,492
README_04_prompt_construction_and_structured_output.md | Pages: 1 | Characters: 9,608
README_05_end_to_end_local_ai_assistant.md | Pages: 1 | Characters: 32,360
README_06_prompt_regression_testing.md | Pages: 1 | Characters: 22,690
README_07_hadota_web_chat.md | Pages: 1 | Characters: 9,497
README_08_HADOTA_Local_AI_Assistant.md | Pages: 1 | Characters: 29,557

FILE TYPE SUMMARY
--------------------------------------------------------------

## Result Interpretation

The implemented extraction components were evaluated against the actual project knowledge base.

### Observed Results

| Metric                     |  Measured Result |
| -------------------------- | ---------------: |
| Discovered documents       |                6 |
| Successfully extracted     |                6 |
| Failed extractions         |                0 |
| Empty documents            |                0 |
| Total logical pages        |                6 |
| Total extracted characters |          118,204 |
| Total source size          |    118,951 bytes |
| Extracted text size        |    118,383 bytes |
| Execution time             | 0.081509 seconds |

### Document-Level Results

| Document                                               | Extracted Pages | Extracted Characters |
| ------------------------------------------------------ | --------------: | -------------------: |
| README_03_Local_Runtime_Engineering.md                 |               1 |               14,492 |
| README_04_prompt_construction_and_structured_output.md |               1 |                9,608 |
| README_05_end_to_end_local_ai_assistant.md             |               1 |               32,360 |
| README_06_prompt_regression_testing.md                 |               1 |               22,690 |
| README_07_hadota_web_chat.md                           |               1 |                9,497 |
| README_08_HADOTA_Local_AI_Assistant.md                 |               1 |               29,557 |

### Key Findings

1. All six discovered Markdown documents were extracted successfully.
2. No extraction failures occurred.
3. No extracted document had empty content.
4. Each Markdown document was represented as one logical page.
5. The total extracted content was 118,204 characters.
6. The extraction experiment completed in approximately 0.082 seconds.
7. The extracted UTF-8 text occupied 118,383 bytes.

### Measurement Considerations

Source file size is measured in bytes, whereas extracted character count measures Unicode characters. These values are not directly interchangeable.

The extracted text size is a separate UTF-8 byte measurement. It should not be treated as proof of exact source preservation without a content-level comparison.

Some documents show differences between their original file sizes and extracted character counts. These differences require a dedicated content-integrity experiment before making conclusions about extraction fidelity.

### Limitations

* The real corpus currently contains Markdown files only.
* TXT and PDF extraction were validated using controlled test fixtures.
* No OCR or scanned-document processing was performed.
* The experiment measured extraction success and size statistics, not semantic completeness.
* Exact content preservation has not yet been verified using source and extracted-content hashes.

## Finding

The extraction components successfully processed all six real knowledge base documents without failures or empty outputs.

The measured execution time was 0.081509 seconds.

The extraction layer is functionally ready for the next stage, subject to a content-integrity check and removal of the deprecated PyMuPDF import.
